# Travel

In [1]:
import datetime
print(f"Last updated: {str(datetime.datetime.now())[:10]}")

Last updated: 2026-10-07


After traveling to various cultural sites for some years, I found a rule of thumb for finding good spots to visit: check if something is a UNESCO World Heritage site. Sometimes this leads to obvious suggestions (Vatican City, Paris, Amsterdam) but more often than not visiting world heritage sites has shown me sites that are very much worth visiting but not as touristy as the usual sites (e.g., Bologna, Orange, Albi).

At first, my enthusiasm led to the dream of visiting all world heritage sites in Europe, but this dream soon showed to be too ambitious. All EU countries then? Still quite a lot. Some narrowing down brought me to the following goal: visit all world heritage sites in the continental part of the original member states of the European Union. Achievable, but also ambitious, with Germany, France and Italy as the biggest obstacles. Since then, I've been using this as a heuristic in planning vacations.

(Of course, I still travel outside of these guidelines as well. Occasionally this leads to another world heritage site, and these are also included in the below map)

Lines mark separate vacations. Lines in grey mark planned vacations, lines in light gray are trips I'd like to make someday

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import numpy as np
from travel_plotter import journey_generator

def customwrap(s, width=10):
    s_split = s.split(' ')
    output = []
    current = ""
    for item in s_split:
        current += item + ' '
        if len(current) > width:
            output.append(current)
            current = ""
    if current != "":
        output.append(current)
    return "<br>".join(output)

def customsplit(s, length=900):
    s_split = s.split('.')
    output = ""
    for sentence in s_split:
        if len(output) + len(sentence) < length and sentence.strip():
            output += sentence + '.'
        else:
            break
    assert len(output) < length
    return output

# Prepare dataset
df = pd.read_csv("WHC-sites.csv")

fig = go.Figure()
for lats, lons, color, year in journey_generator(df):
    fig2 = px.line_map(lat=lats, lon=lons, color_discrete_sequence=[color])
    fig2.update_layout(map_style='carto-positron', hovermode=False)
    fig = go.Figure(data=fig.data + fig2.data)

df = df[df['years_visited'].notna()]
df['year_last_visited'] = df['years_visited'].apply(lambda x: x[-4:] if not x == "-1" else "Not yet visited")
df["year_last_visited_text"] = df["year_last_visited"].apply(lambda x: f"Last visited in: {x}<br>" if x != "Not yet visited" else "")
df['rating'] = df['rating'].apply(lambda x: x if pd.notna(x) else "Not yet visited")
df["rating_text"] = df["rating"].apply(lambda x: f"Rating: {x}<br>" if x != "Not yet visited" else "Not yet visited<br>")
df['visited'] = (df['year_last_visited'] != 'Not yet visited')

df['name_en'] = df['name_en'].apply(lambda x: customwrap(x, 40))
df['short_description_en'] = df['short_description_en'].apply(lambda x: customsplit(customwrap(x, 40)))
assert not df['short_description_en'].apply(lambda x: len(x) > 900).any()

max_bound = max(abs(min(df['latitude'].astype(float))-max(df['latitude'])), abs(min(df['longitude'])-max(df['longitude']))) * 175
zoom = 11.25 - np.log(max_bound)


fig1 = px.scatter_map(df, lat='latitude', lon='longitude', color='visited',
                       hover_name='name_en', map_style='carto-positron', zoom=zoom,
                       center = {'lat':df['latitude'].mean(), 'lon':df['longitude'].mean()},
                       custom_data=['rating_text', 'year_last_visited_text', "short_description_en"], hover_data={
                           'latitude':False,
                           'longitude':False,
                       })
fig1.update_traces(hovertemplate='<b>%{hovertext}</b><br>%{customdata[0]}%{customdata[1]}<br>%{customdata[2]}<extra></extra>')
fig1.update_layout(showlegend=False, margin={'t':0,'l':0,'b':0,'r':0})

fig = go.Figure(data=fig.data + fig1.data, layout=fig1.layout)

config = {'scrollZoom': True}
fig.show(config=config)


<p style="text-align: center;">↑ Click me!</p>

And how is that going?

In [3]:
import numpy as np

df1 = df[
    (df['iso_code'].str.contains('nl')) |
    (df['iso_code'].str.contains('fr')) |
    (df['iso_code'].str.contains('it')) |
    (df['iso_code'].str.contains('be')) |
    (df['iso_code'].str.contains('de')) |
    (df['iso_code'].str.contains('lu'))
]

abbrs = ["Total",'be', 'de', 'fr', 'it', 'lu', 'nl' ]
countries = ["Total", "Belgium", "Germany", "France",  "Italy", "Luxembourg", "Netherlands",]
got_arr = []; need_arr = []
percentages = []

for abbr in abbrs:
    got = len(df1[df1['visited'] & (df1['iso_code'].str.contains(abbr))])
    need = len(df1[df1['iso_code'].str.contains(abbr)])
    if abbr == "Total":
        got = len(df1[df1['year_last_visited'] != 'Not yet visited'])
        need = len(df1)
    got_arr.append(got); need_arr.append(need)
    percentages.append((got/need) * 100)

colors = ["complete_color" if perc == 100 else "incomplete_color" for perc in percentages]
colors[0] = "total_color" if colors[0] == "incomplete_color" else "complete_color"

customdata = [got_arr, need_arr]
d = {"Countries": countries, "Progress": percentages}
df1 = pd.DataFrame(data=d)
fig = px.bar(df1, x="Countries", y="Progress", template='plotly_white', color=colors,
             color_discrete_map={"total_color":"#ef553b", "complete_color":"#FECB52", "incomplete_color":"#636efa"},
             custom_data=customdata)
fig.update_traces(hovertemplate='%{x} progress:<br>%{customdata[0]} out of %{customdata[1]}<extra></extra>')
fig.update_layout(yaxis_range=[0,100], showlegend=False)
fig.show()

Of course, new sites are added each year, so I do need to average a certain amount of sites per year to 'keep track'

In [4]:
df['year_first_visited'] = df['years_visited'].apply(lambda x: int(x[:4]) if not x == "-1" else 0)
df2 = df[
    (df['iso_code'].str.contains('nl')) |
    (df['iso_code'].str.contains('fr')) |
    (df['iso_code'].str.contains('it')) |
    (df['iso_code'].str.contains('be')) |
    (df['iso_code'].str.contains('de')) |
    (df['iso_code'].str.contains('lu'))
]

# Cumulative sum of visits to WHS in years
visited_years = df2[df2['year_first_visited'] != 0]
visited_years = visited_years['year_first_visited'].value_counts().sort_index().cumsum()
visited_years = pd.DataFrame(visited_years)
# Cumulative sum of inscribed WHS in years
visited_years['type'] = ['Amount of WHS visited'] * len(visited_years)
inscribed_years = df2['date_inscribed'].value_counts().sort_index().cumsum()
inscribed_years = pd.DataFrame(inscribed_years)
inscribed_years['type'] = ['Amount of WHS inscribed'] * len(inscribed_years)
# Combine to one dataframe and plot
combined_years = pd.concat([visited_years, inscribed_years], axis=0)
combined_years = combined_years[combined_years.index>2010]
fig = px.line(combined_years, y='count', color='type', template='plotly_white', 
            labels={"index": "Year", "count": "Amount", 'type':''},
        )
fig.update_traces(hovertemplate="%{x}: %{y}")
fig.update_layout(legend = dict(orientation="h", yanchor='bottom', y= 1.02, xanchor='right', x=0.55))
fig.show()
